<a href="https://colab.research.google.com/github/yunayana/Sztuczna_Inteligencja_STAC_2026/blob/main/Lab4_plik_z_danymi_cwiczenie1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn import preprocessing
raw_csv_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')
unscaled_inputs_all = raw_csv_data[:,1:-1 ]
targets_all = raw_csv_data[:,-1]

In [2]:
shuffled_indices = np.arange(unscaled_inputs_all.shape[0])
np.random. shuffle(shuffled_indices)

unscaled_inputs_all = unscaled_inputs_all[shuffled_indices]
targets_all = targets_all[shuffled_indices]

In [3]:
# Policzymy, ile jest targetów 1 (co oznacza, że klient wrócił do sklepu)
num_one_targets = int(np.sum(targets_all))

# Ustaw licznik dla targetów, które wynoszą 0
#(oznaczających, że klient nie wrócił do sklepu)
zero_targets_counter = 0

# Chcemy utworzyć „zrównoważony” zbiór danych,
# więc będziemy musieli usunąć niektóre pary wejście/target
# by targetów 0 i 1 było mniej więcej tyle samo.
# Zadeklarujemy zmienną, która to zrobi:
indices_to_remove = []

# Policzymy liczbę targetów, które są równe 0.
# Gdy liczba targetów z 0 będzie równa liczbie targetów z 1,
# zaznaczymy pozostałe wpisy target 0 do usunięcia.
for i in range(targets_all.shape[0]):
  if targets_all[i] == 0:
    zero_targets_counter += 1
    if zero_targets_counter > num_one_targets:
        indices_to_remove.append(i)

# Utwórzymy dwie nowe zmienne:
# - jedną, która będzie zawierać dane wejściowe
# - jedną, która będzie zawierać targety.
# Usuwamy wszystkie indeksy, które oznaczyliśmy jako „do usunięcia” w pętli powyżej.
unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

In [4]:
# To jedyne miejsce, w którym używamy funkcjonalności sklearn.
# Wykorzystamy jej możliwości wstępnego przetwarzania.
# To prosta linia kodu, która normalizuje dane wejściowe.
# Pod koniec pracy możesz spróbować uruchomić algorytm BEZ tej linii kodu.
# Wynik będzie interesujący.
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

In [5]:
# Przetasujemy ponownie indeksy danych,
# aby dane nie były w żaden sposób uporządkowane, kiedy je wprowadzamy.
# Ponieważ będziemy przetwarzać partiami,
# chcemy, aby dane były rozłożone tak losowo, jak to możliwe
# Tym razem robimy to z danymi już znormalizowanymi
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random. shuffle(shuffled_indices)

# Użyjemy przetasowanych indeksów, aby przetasować dane wejściowe i docelowe.
shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

In [7]:
# Policzymy całkowitą liczbę próbek
samples_count = shuffled_inputs.shape[0]

# Policzymy próbki w każdym podzbiorze, zakładając,
# że chcemy rozkładu 80%-10%-10% treningu, walidacji i testu.
# Oczywiście liczby są liczbami całkowitymi.
train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)

# Zestaw danych „testowy” zawiera wszystkie pozostałe dane.
test_samples_count = samples_count - train_samples_count - validation_samples_count

# Utwórzymy zmienne, które rejestrują dane wejściowe i cele do treningu
# W naszym przetasowanym zestawie danych są to pierwsze obserwacje „train_samples_count"
train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

# Utwórzymy zmienne, które rejestrują dane wejściowe i docelowe do walidacji.
# Są to kolejne obserwacje „validation_samples_count",
# następujące po „train_samples_count", które już przypisaliśmy
validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count: train_samples_count+validation_samples_count]

# Utwórzymy zmienne, które rejestrują dane wejściowe i targety testu.
# Są wszystkim, co pozostało.
test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count: ]

In [8]:
# Zrównoważyliśmy nasz zbiór danych w stosunku 50-50 (dla celów 0 i 1),
# ale trening, walidacja i test zostały pobrane z pomieszanego zbioru danych.
# Sprawdzimy, czy są również zrównoważone.
# Zwróć uwagę, że za każdym razem, gdy ponownie uruchamiasz ten kod,
# otrzymasz inne wartości, ponieważ za każdym razem są one losowo pomieszane.
# Zwykle wstępnie przetwarzasz RAZ, więc nie musisz ponownie uruchamiać tego kodu po jego zakończeniu.
# Jeśli ponownie uruchomisz cały arkusz,
# pliki datasetu .npz zostaną nadpisane nowo przetworzonymi wstępnie danymi.

# Wydrukuj liczbę celów, które są 1, całkowitą liczbę próbek i proporcję dla treningu, walidacji i testu.
print(np.sum(train_targets), train_samples_count, np.sum(train_targets) / train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets) / test_samples_count)

1797.0 3579 0.5020955574182733
218.0 447 0.48769574944071586
222.0 448 0.4955357142857143


In [9]:
# Zapiszemy trzy zestawy danych w formacie *. npz.
# Niezwykle ważne jest nadawanie im spójnych nazw!

np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)